#STEP 0: PREPROCESSING

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

In [ ]:
df = pd.read_csv('/content/star_classification.csv')
print("Shape:", df.shape)
df.head()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
print(df['class'].value_counts())
sns.countplot(x='class', data=df)
plt.title('Class Distribution - Galaxy vs Star vs QSO')
plt.show()

In [ ]:
df = df[(df[['u', 'g', 'r', 'i', 'z']] != -9999).all(axis=1)].copy()

In [ ]:
cols_to_drop = ['obj_ID', 'run_ID', 'rerun_ID', 'cam_col', 'field_ID', 'spec_obj_ID', 'plate', 'MJD', 'fiber_ID']
df = df.drop(columns=cols_to_drop)

X = df.drop('class', axis=1)
y = df['class']

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [ ]:
for col in X.columns:
    cap = X[col].quantile(0.99)
    floor = X[col].quantile(0.01)
    X[col] = X[col].clip(lower=floor, upper=cap)

In [ ]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [ ]:
print("Final X_scaled shape:", X_scaled.shape)
print("Final y_encoded shape:", y_encoded.shape)

print("\nTotal missing values in X_scaled:", X_scaled.isnull().sum().sum())

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

#MODEL #1: RANDOM FOREST

In [ ]:
rf_model1 = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced')
rf_model1.fit(X_train, y_train)
y_pred1 = rf_model1.predict(X_test)

print("=== Random Forest - No Tuning ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred1):.4f}")
print(classification_report(y_test, y_pred1, target_names=['GALAXY', 'QSO', 'STAR']))

In [ ]:
cm1 = confusion_matrix(y_test, y_pred1)
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues',
            xticklabels=['GALAXY', 'QSO', 'STAR'],
            yticklabels=['GALAXY', 'QSO', 'STAR'])
plt.title('Confusion Matrix - No Tuning')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [None, 20],
    'max_features': ['sqrt']
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_)

In [ ]:
y_pred2 = grid_search.best_estimator_.predict(X_test)

print("=== Random Forest - With Tuning ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred2):.4f}")
print(classification_report(y_test, y_pred2, target_names=['GALAXY', 'QSO', 'STAR']))

In [ ]:
cm2 = confusion_matrix(y_test, y_pred2)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues',
            xticklabels=['GALAXY', 'QSO', 'STAR'],
            yticklabels=['GALAXY', 'QSO', 'STAR'])
plt.title('Confusion Matrix - With Tuning')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

#MODEL #2: XGBOOST

In [ ]:
from xgboost import XGBClassifier

XGBOST_MODEL = XGBClassifier(random_state=42, class_weight='balanced')
XGBOST_MODEL.fit(X_train, y_train)
y_pred1 = XGBOST_MODEL.predict(X_test)

print("=== XGBOOST MODEL - No Tuning ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred1):.4f}")
print(classification_report(y_test, y_pred1, target_names=['GALAXY', 'QSO', 'STAR']))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm_base = confusion_matrix(y_test, y_pred1)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues',
            xticklabels=['GALAXY', 'QSO', 'STAR'],
            yticklabels=['GALAXY', 'QSO', 'STAR'])
plt.title('Baseline XGBoost')
plt.show()

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

param_distributions = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2]
}

random_search = RandomizedSearchCV(estimator=XGBClassifier(random_state=42,n_jobs=-1, eval_metric='mlogloss'), param_distributions=param_distributions,
    n_iter=10,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)
print("Best Cross-Validation Score:", random_search.best_score_)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

y_pred_tuned = random_search.predict(X_test)
cm_tuned = confusion_matrix(y_test, y_pred_tuned)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Greens',
            xticklabels=['GALAXY', 'QSO', 'STAR'],
            yticklabels=['GALAXY', 'QSO', 'STAR'])
plt.title('Tuned XGBoost')
plt.show()

code to make sure models are not overfitted and preprocessing was applied ccorrectly

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf_model1, X_scaled, y_encoded, cv=5, scoring='accuracy')

print("Scores for each fold:", scores)
print("Average CV Accuracy:", scores.mean())
print("Score standard deviation:", scores.std())

# MODEL #3: LIGHTGBM

In [ ]:
!pip install lightgbm -q

from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
X_raw = df.drop('class', axis=1).copy()
y_raw = le.transform(df['class'])

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

In [ ]:
lgbm_raw = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    random_state=42,
    verbose=-1
)

lgbm_raw.fit(X_train_raw, y_train_raw)
y_pred_lgbm_raw = lgbm_raw.predict(X_test_raw)

print("=== LightGBM Without Preprocessing ===")
print("Accuracy:", accuracy_score(y_test_raw, y_pred_lgbm_raw))
print("Precision:", precision_score(y_test_raw, y_pred_lgbm_raw, average='macro'))
print("Recall:", recall_score(y_test_raw, y_pred_lgbm_raw, average='macro'))
print("F1-score:", f1_score(y_test_raw, y_pred_lgbm_raw, average='macro'))
print(classification_report(y_test_raw, y_pred_lgbm_raw, target_names=le.classes_))

In [ ]:
cm_lgbm_raw = confusion_matrix(y_test_raw, y_pred_lgbm_raw)

sns.heatmap(
    cm_lgbm_raw,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)

plt.title("Confusion Matrix - LightGBM Without Preprocessing")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
lgbm_preprocessed = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    random_state=42,
    verbose=-1
)

lgbm_preprocessed.fit(X_train, y_train)
y_pred_lgbm_preprocessed = lgbm_preprocessed.predict(X_test)

print("=== LightGBM With Preprocessing ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lgbm_preprocessed))
print("Precision:", precision_score(y_test, y_pred_lgbm_preprocessed, average='macro'))
print("Recall:", recall_score(y_test, y_pred_lgbm_preprocessed, average='macro'))
print("F1-score:", f1_score(y_test, y_pred_lgbm_preprocessed, average='macro'))
print(classification_report(y_test, y_pred_lgbm_preprocessed, target_names=le.classes_))

In [ ]:
cm_lgbm_preprocessed = confusion_matrix(y_test, y_pred_lgbm_preprocessed)

sns.heatmap(
    cm_lgbm_preprocessed,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)

plt.title("Confusion Matrix - LightGBM With Preprocessing")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
param_grid_lgbm = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [5, 10, 15],
    'num_leaves': [31, 63, 127]
}

lgbm_search = RandomizedSearchCV(
    estimator=LGBMClassifier(
        objective='multiclass',
        num_class=3,
        random_state=42,
        verbose=-1
    ),
    param_distributions=param_grid_lgbm,
    n_iter=10,
    cv=3,
    scoring='f1_macro',
    random_state=42,
    n_jobs=-1
)

lgbm_search.fit(X_train, y_train)

best_lgbm = lgbm_search.best_estimator_

print("Best LightGBM Parameters:")
print(lgbm_search.best_params_)
print("Best Cross-Validation Score:", lgbm_search.best_score_)

In [ ]:
y_pred_lgbm_tuned = best_lgbm.predict(X_test)

print("=== Tuned LightGBM ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lgbm_tuned))
print("Precision:", precision_score(y_test, y_pred_lgbm_tuned, average='macro'))
print("Recall:", recall_score(y_test, y_pred_lgbm_tuned, average='macro'))
print("F1-score:", f1_score(y_test, y_pred_lgbm_tuned, average='macro'))
print(classification_report(y_test, y_pred_lgbm_tuned, target_names=le.classes_))

In [ ]:
cm_lgbm_tuned = confusion_matrix(y_test, y_pred_lgbm_tuned)

sns.heatmap(
    cm_lgbm_tuned,
    annot=True,
    fmt='d',
    cmap='Oranges',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)

plt.title("Confusion Matrix - Tuned LightGBM")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
lgbm_accuracy = accuracy_score(y_test, y_pred_lgbm_tuned)
lgbm_precision = precision_score(y_test, y_pred_lgbm_tuned, average='macro')
lgbm_recall = recall_score(y_test, y_pred_lgbm_tuned, average='macro')
lgbm_f1 = f1_score(y_test, y_pred_lgbm_tuned, average='macro')

#PROBLEM REFRAMING

#KAGGLE Notebook Coparison

---



Selected Kaggle Notebook:  
https://www.kaggle.com/code/navdeepjakhar/stellar-classification-comparing-6-algorithms

For this part of the project, a public Kaggle notebook based on the same Stellar Classification dataset was reviewed and compared with our implementation. The selected notebook applied several machine learning algorithms to classify stars, galaxies, and quasars using astronomical and spectral features.

The public notebook provided a useful reference for understanding common machine learning approaches used for stellar classification. Similar to our project, the notebook focused on multiclass classification and applied machine learning models to astronomical data. However, our implementation followed a more structured experimental methodology and placed greater emphasis on model comparison, preprocessing analysis, and evaluation consistency.

One important difference was that our project evaluated multiple models using the same experimental conditions and evaluation metrics. Random Forest, XGBoost, and LightGBM were compared using accuracy, precision, recall, F1-score, and confusion matrices. In addition, LightGBM was tested under multiple configurations, including training on raw data, training on preprocessed data, and training with tuned hyperparameters. This allowed a more detailed evaluation of how preprocessing and optimization affected predictive performance.

Another difference involved the interpretation of experimental results. In our implementation, confusion matrices were analyzed carefully to understand classification behavior between stars, galaxies, and quasars. This provided deeper insight into model strengths and weaknesses rather than relying only on final accuracy values.

The comparison also highlighted the importance of preprocessing and hyperparameter tuning. While the public notebook demonstrated effective baseline classification techniques, our project included additional experimentation stages that made the workflow more comprehensive and easier to analyze systematically.

Overall, reviewing the public Kaggle notebook helped strengthen our understanding of stellar classification techniques and allowed us to reflect on the strengths of our own implementation. The comparison demonstrated that our project provided a more organized experimental workflow, more detailed evaluation procedures, and a clearer comparison between different machine learning models.

# Problem Re-framing (Removing Stars)

In [ ]:
X_ds = X_scaled[y_encoded != 2]
y_ds = y_encoded[y_encoded != 2]

X_train_ds, X_test_ds, y_train_ds, y_test_ds = train_test_split(
    X_ds, y_ds, test_size=0.2, random_state=42
)

lgbm_ds = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=10,
    num_leaves=63,
    random_state=42
)

lgbm_ds.fit(X_train_ds, y_train_ds)

y_pred_ds = lgbm_ds.predict(X_test_ds)

print("--- Deep Space Classification (No Stars) ---")
print(classification_report(y_test_ds, y_pred_ds, target_names=['GALAXY', 'QSO']))
ConfusionMatrixDisplay.from_predictions(y_test_ds, y_pred_ds, display_labels=['GALAXY', 'QSO'])
plt.show()